## Central Question and Limitations
- What role do wetlands play in wildfire movement across the landscape?
- Data limitations: restricted to fires with available spatial data in the form of a vector perimeter, begining with the year 2025

# Goals
- Determine how much overlap in wildfire acreage and wetland acreage exists
- Determine how many wildfires occured within 1 (?) miles of a wetland

In [1]:
# import packages
import platform
from pathlib import Path
import pandas as pd
import geopandas as gpd
import seaborn as sns
import calendar
import matplotlib.pyplot as plt
import pygris

In [2]:
# import cleaned fire dataset

project_fldr = Path.cwd().parents[1]

raw_fldr = project_fldr / 'data' / 'raw'
processed_fldr = project_fldr / 'data' / 'processed'

# read in CSV
fire_shp_25 = gpd.read_file(processed_fldr / "fire_shp_25/fire_shp_25.shp")
fire_shp_25.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 521 entries, 0 to 520
Data columns (total 23 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   YEAR_       521 non-null    float64 
 1   STATE       521 non-null    str     
 2   AGENCY      521 non-null    str     
 3   UNIT_ID     521 non-null    str     
 4   FIRE_NAME   521 non-null    str     
 5   INC_NUM     521 non-null    str     
 6   IRWINID     469 non-null    str     
 7   ALARM_DATE  521 non-null    str     
 8   CONT_DATE   514 non-null    str     
 9   C_METHOD    521 non-null    int32   
 10  CAUSE       521 non-null    int32   
 11  OBJECTIVE   521 non-null    float64 
 12  GIS_ACRES   521 non-null    float64 
 13  COMPLEX_ID  17 non-null     str     
 14  COMPLEX_NA  17 non-null     str     
 15  COMMENTS    120 non-null    str     
 16  FIRE_NUM    0 non-null      object  
 17  Shape_Leng  521 non-null    float64 
 18  Shape_Area  521 non-null    float64 
 19  

In [3]:
# and we can't store attributes as anything except strings in shp
# but we at least want dates to be in datetime right now
fire_shp_25["ALARM_DATE"] = pd.to_datetime(fire_shp_25["ALARM_DATE"], format='ISO8601')
fire_shp_25["CONT_DATE"] = pd.to_datetime(fire_shp_25["CONT_DATE"], format='ISO8601')



In [4]:
# import cleaned wetlands dataset
# establish paths to datasets - wetlands

# set a path to the main box folder based on machine's os
# if windows: 
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
# if mac:
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#Confirm that the box path is valid
if box_path.exists():
    print(f'Box folder found at {box_path}')
else:
    print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

# create path object that goes to the specific folder on Box drive
wetlands_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'wetlands_subset' / 'wetlands_subset.geojson'
print(wetlands_path.exists())

# read in the wetlands geojson file, this will take a WHILE - 10 mins 1st try 
wetlands = gpd.read_file(wetlands_path, engine="pyogrio")

Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True


In [6]:
# make sure both shapefiles are in the same CRS
print(fire_shp_25.crs)
print(wetlands.crs)

# put them in the same CRS - and make sure it's projected
# EPSG:26910 for UTM zone 10 in CA
fire_crs = fire_shp_25.to_crs("EPSG:26910")
wetlands_crs = wetlands.to_crs("EPSG:26910")

print(fire_crs.crs)
print(wetlands_crs.crs)


EPSG:3310
EPSG:5070
EPSG:26910
EPSG:26910


In [ ]:
# now we want to see for each wetland, what is the distance to the closest fire
nearest_fires = nearest_fires = gpd.sjoin_nearest(
    wetlands_crs, fire_crs, how="left", distance_col="Dist_Closest_Fire"
)